In [ ]:
import sphere_ref_lib as srl
import sphere_variables as sv

import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd

fp = srl.set_output_dir()
fp_nc = srl.set_output_dir_nc()
R, step, xs, d, Z0, R2 = srl.set_basic_params()
R2s, R2, target, apply_ref, _, _ = sv.svv()
e1, e2, H_obs, D_moon, R_moon, e1, e2, tandelta = srl.get_default_param(target)
theta_arr_r2, phi_arr_r2, p0s, p2s, p_hits, ref_dirs, amp_arr, alp_arr = srl.ref_rays_count_ref(R2, xs, d, Z0, R, y=0.0, print_info=True, reflectance=False, e1=e1)
if not os.path.exists(fp_nc + "alpha_to_theta_forhist_ver2.nc"):
    R2s_make_nc = np.array(R2s)
    srl.make_nc_alpha_ts(R2s_make_nc*1000, fp_nc, fn = "alpha_to_theta_forhist_ver2.nc")

plot_colors = ["red", "darkorange", "springgreen", "mediumblue", "fuchsia"]

In [ ]:
param_list, cor_list = [], []

for i in range(len(R2s)):

    R2 = R2s[i]

    params = {
        "target" : target,
        "R2" : R2,
        "fp_nc" : fp_nc,
        "theta_arr_r2" : theta_arr_r2,
        "phi_arr_r2" : phi_arr_r2,
        "p0s" : p0s,
        "p2s" : p2s,
        "p_hits" : p_hits,
        "ref_dirs" : ref_dirs,
        "amp_arr" : amp_arr,
        "alp_arr" : alp_arr,
    }

    cors = [
        #("solid_angle", {}),
        ("solid_angle_field", {}),
        ("reflection_rate", {"apply_ref": "average", "fn": "alpha_to_theta_forhist_ver2.nc"}),
        #("vertical_direction", {}),
    ]
    param_list.append(params)
    cor_list.append(cors)

In [ ]:
ds_all = srl.load_nc_sphere(1.8, fp_nc)
heat_da = srl.make_thph_2dhist(ds_all)

fig, ax = plt.subplots(figsize=(8, 6))

fin_da = srl.steve_correction_pipeline(heat_da, params, cors)

fin_da.plot(ax=ax, x="theta", y="phi", cmap="viridis", vmin=0.0, vmax=600)

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))

ax.grid()

for i in range(len(R2s)):

    R2 = R2s[i]
    params = param_list[i]
    cors = cor_list[i]

    ds_all = srl.load_nc_sphere(R2, fp_nc)

    heat_da = srl.make_thph_2dhist(ds_all)

    fin_da = srl.steve_correction_pipeline(heat_da, params, cors)

    phi_min, phi_max = 0.0, 90.0

    fin_da_090 = fin_da.where((fin_da["phi"]>=phi_min) & (fin_da["phi"]<=phi_max), drop=True)
    th_1dcounts_090 = fin_da_090.sum(dim="phi")
    th_1dcounts_090.name = "counts (sum over phi 0-90)"

    
    ax.plot(th_1dcounts_090["theta"].values, th_1dcounts_090.values, c=plot_colors[i], label=f"R2={R2}")
    ax.set_title(f"Counts vs Theta (summed over phi=0-90)")
    ax.set_xlabel("Theta (degrees)")
    ax.set_ylabel("Counts (sum over phi 0-90)")
    #ax.set_xlim(90, 100)
    #ax.set_ylim(0,20000)

inc_th = np.radians(np.arange(0,180,0.1))
inc_num = np.cos(inc_th) / step
ax.plot(inc_th, inc_num, c="black", ls="--", label="Incident beam")

plt.legend()
plt.show()

In [ ]:
dth_da = srl.load_nc_alpha_ts(fp_nc, fn="alpha_to_theta_forhist_ver2.nc")

pd.DataFrame(dth_da, index=dth_da["alpha"], columns=dth_da["H"]).loc[90:100,:]